environment delpher

In [1]:
import os
import json
from lxml import etree
import requests

##### Handle single alto files

In [2]:
file_path = '../../../DATA/europeans-news/alto/urn=ddd_000023649_mpeg21_p002_alto.alto.xml'
filename = os.path.basename(file_path)
file_name = ''.join(os.path.splitext(filename)[0].split('.')[:-1])
file_name = file_name.replace('_', ':')
print(file_name)

urn=ddd:000023649:mpeg21:p002:alto


In [3]:
from gliner import GLiNER

labels = ["person", "location (geographical)", "organization"]

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
model.eval()
print("ok")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/traitlets

ok


In [4]:
def spans_to_bio(tokens, spans):
    # tokens: list[str]
    # spans: list[{"start": int, "end": int, "label": str}]

    token_dict = {}
    index = 0
    for i, token in enumerate(tokens):
        token_dict[i] = {"token": token, "offset": index, "label": "O"}
        index += len(list(token)) + 1

    # Update labels based on spans
    for span in spans:
        span_start = span["start"]
        span_end = span["end"]
        span_label = span["label"]
        
        first = True
        for token_idx in token_dict:
            token_offset = token_dict[token_idx]["offset"]
            # Check if token offset falls within span range
            if token_offset >= span_start and token_offset < span_end:
                if first:
                    token_dict[token_idx]["label"] = f"B-{span_label[:3].upper()}"
                    first = False
                else:
                    token_dict[token_idx]["label"] = f"I-{span_label[:3].upper()}"
    
    return token_dict

In [5]:
def split_long_sentences(text):
    """
    Check if any sentence exceeds 350 characters.
    If yes, break it into smaller chunks (<350) and return updated text.
    """
    # Split by periods to get sentences
    sentences = text.split('.')
    result_sentences = []
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        # Check if sentence exceeds 350 characters
        if len(sentence) > 350:
            # Break into chunks
            words = sentence.split()
            current_chunk = []
            current_length = 0
            
            for word in words:
                # If adding next word exceeds 350, save current chunk
                if current_length + len(word) + 1 > 350 and current_chunk:
                    result_sentences.append(" ".join(current_chunk))
                    current_chunk = [word]
                    current_length = len(word)
                else:
                    current_chunk.append(word)
                    current_length += len(word) + 1
            
            # Add remaining words
            if current_chunk:
                result_sentences.append(" ".join(current_chunk))
        else:
            # Sentence is fine, keep as is
            result_sentences.append(sentence)
    
    # Join sentences back with periods
    return ". ".join(result_sentences) + "."

In [11]:
def create_block_of_text(page_tree):
    ns = {
            "alto": "http://schema.ccs-gmbh.com/ALTO"
        }
    block_of_text = []
    for text_block in page_tree.findall('.//alto:TextBlock', namespaces=ns):
        id = text_block.get("ID")
        text = []
        wc_score = []
        for string in text_block.findall('.//alto:String', namespaces=ns):
            text.append(string.get("CONTENT"))
            wc_score.append(float(string.get("WC")))
        block_of_text.append({
            "id": id,
            "text": " ".join(text),
            "wc_score": sum(wc_score) / len(wc_score) if wc_score else 0
        })
    return block_of_text

In [27]:
text = """
 Koning zeer ongunstige tijdingen ontvangen ; zlj te vatten de mededeelinp', dat de levenskrachten sterk afnemen, dat er een algemeen lijdende toestand is ingetreden, terwijl zich eene groote verzwakking in de beenen heeft voorgedaan. Dientengevolge is voorloopig tegenbevel len aanzien der terugkomst van het kasteel der Ardennen naar hier gegeven " Eene korre«pondentie uit Rome deelt de instrukliën mede welke het revolutionnair zoogenaamd Romeinsch komité tot zijne aanhanaers geriït heeft Deze in strukliën moeten de oogen der ongeloovigsten doen opengaan, daar zij voor elkeen duidelijk doen uitko men dat de Italiaansche revolulionnairen voornemens zijn het Pauselijk gezag te vernietigen zoodra de Fransche troepen uit Rome zullen vertrokken zijn. Hoe meer de dag van de opening der zitting van de. Italiaansche Kamers nadert, hoe meer de ongele genheden aangro*ijen onder de voeten van het minis terie. He generaal La Marmora, minister-president, had, dm 12en, nog geenen Voorzitter van den Senaat kunnen vinden. Hij had zich achtereenvolgens tot dea markies Gino Caypmi en den graaf Casati gewend, maar beiden hebben geweigerd, omdat zij zich niet willen veieenigen met de maatregelen, welke door het gouvernement voorbereid zijn tegen de geestelijke korpora' i Er zullen, volgens een berigt van het Journal des Débats, in Italië eerstdaags nog ruim honderd ver ki zingen voor de Kamer van Afgevaardigden moe ten plaats hebben, daar een aantal van de gekozenen d|i twee of meer plaatsen de meerderheid bekomen of de keuze niet aangenomen hebben, en daar ook eenige keuzen wegers onregelmatigheden vernietigd zullen ten worden. Men kan zich voorstellen welke ab- D' rn nle verkiezingen en welk vreemd mengelmoes van een.- Kamer het moeten zijn.waar bij de eerste verkiezing slechts ruim honderd kandidaten gekozen werden en ook na de herstemming nog meer dan honderd nieuwe verkiezingen moeten plaats hebben. In Frankrijk zal het leger in getalsterkte vermin derd « orde u ; vooral zal de vermindering plaats heb ben in de keizerlijke garde. Daarentegen zullen de traktementen der luitenants en kapiteins waarschijn lijk worden verhoogd. Ook in Spanje zullen de uitgaven voor het leger worden verminderd*. 8 man van elke kompagnie in fanterie, en 4 ruiters per eskadron zullen den 20n met verlof gezonden worden. De jongste berigten uit Madrid brengen den in houd van een besluit der Koningin, hetwelk bewijst dat het vrijhandelsstelsel ook in Spanje begint door te dringen. Daarbij is namelijk eene staatskommis sie benoemd om te onderzoeken of en in hoever de opheffing van differenlieele invoerregten in Spanje wenschelijk zou zijn, en in het algemeen, welke be zwaren aan de ontwikkeling der Spaaiibche haudels scheepvaart in den weg staan. Volgens de Weener Neue Trei-Presse zet het Oos tenrijksch gouvernement met knicht de onderhande lingen voort betreffende het sluiten van een handels traktaat met Engeland. Wij hebben vroeger gemeld dat tusschen Engeland en de Vereenigde Staten van Amerika oneenigheden gerezen waren omtrent de invrijheidstelling door eerst genoemd gouvernement van den kapitein en de be manning van het kaperschip Shedenoah. De London Gazette deelt nu''de laatste depêches mede tusschen het Engelsch ministerie van Buitenlandsche Zaken en den heer Adams, Amerikaansch gezant te Londen, gewisseld. De zaken schijnen zeer verzacht. Het Amerikaansch gouvernement dringt niet aan op de scheidsregterlijke uitspraak, welke het eerst vor derde, en zoo het de door Engeland voorgestelde gemengde kommissie nog niet aanneemt, vraagt het ten minste welke vraagstukken haar zullen onder worpen worden, en volgens welke beginselen zij zal moeten werken. Graaf Russell belooft een anl' oord en, in afwachting wentit hij de grootste zorg aan om goed vast te stellen dat het Engelsch gouvernement nimmer van de internationale voorschriften afgeweken is. Alweder een nieuw ministerie in Griekenland. De Fransche Moniteur van gister deelt toch een telegram uit Athene mede, volgens hetwelk het ministerie Delergis, dat slechts een paar weken leven telde, afgetreden eu de heer Bulgaris met de zamenstelling van ien nieuw kabinet belast is. Voorts werd ge dat graaf. Sponneck zou vertrekken. Alzoo de ratfdsman van den jongen koning moet eerst ver rd worden en dan zal de vorst later de 3te kelrker volgen en de reis naar Kopenhagen ondernemen, gelukkig als hij er heelhuids van afkomt, gelijk de vorige koning Otto, die thans op zijne lauweren, in zijn geboorteland Beijeren, leeft. V-algens <h berigten uit Londen is in de laatste inistenaad het vraagstuk van de parle mentaire tiervorming behandeld, maar is dienaan-
   """

In [ ]:
def chunk_text_by_word_limit(text, word_limit=384):
    """
    Split text into chunks respecting the word limit while not breaking sentences.
    """
    sentences = text.split('.')
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        
        sentence_words = sentence.split()
        sentence_word_count = len(sentence_words)
        
        # If adding this sentence would exceed limit and we have content, start new chunk
        if current_word_count + sentence_word_count > word_limit and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_word_count = sentence_word_count
        else:
            current_chunk.append(sentence)
            current_word_count += sentence_word_count
    
    # Add remaining chunk
    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')
    
    return chunks


if len(text.split()) > 384:
    # divide text into chunks of max 384 words while preserving sentence boundaries
    chunks = chunk_text_by_word_limit(block_of_text, word_limit=384)
    print(f"Text split into {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1}: {len(chunk.split())} words")


Sentence is fine: 26
Sentence is fine: 109
Sentence is fine: 112
Sentence is fine: 111
Sentence is fine: 134
Sentence is fine: 196
Sentence is fine: 157
Sentence is fine: 73
Sentence is fine: 186
Sentence is fine: 139
Sentence is fine: 146
Sentence is fine: 50
Sentence is fine: 125
Sentence is fine: 262
Sentence is fine: 143
Sentence is fine: 186
Sentence is fine: 228
Sentence is fine: 141
Sentence is fine: 146
Sentence is fine: 267
Sentence is fine: 153
Sentence is fine: 54
Sentence is fine: 114
Sentence is fine: 230


In [20]:
sentences

['\n j gaande nog niets beslist',
 ' Lord Elcho heeft de be noeming voorgesteld van eene kommissie, welke dit belangrijk vraagstuk zou bestudeeren',
 ' Buiten dit vraagstuk hield de openbare meening zich vooral bezi\n   g met de gebeurtenissen van Canada en Jamaica',
 ' De Mor ning Post hoopt dat Amerika He toebereidselen van eenen inval in Canada door de fenéans, zal verhinderen',
 ' Van een anderen kant verwachtte men met veel angst \n   te Londen de aan komst van de mail, welke tijdingen uit Jamaica zal medebrengen',
 ' Met de in Southampton aangekomen Tasmanian zijn berigten uit Jamaica ontvangen, volgens welke de uitbarsting van den opstand voorb\n    arig was geweest en eerst met Kersmis had moeten plaatshebben',
 ' De opstand werd met kracht onderdrukt, de meeste der gevangen genomen hoofden van den opstand waren dood geschoten, en de nist was bijna overal hers\n    teld',
 ' De opstand der zwarten vnn Jamaica was op de westelijke kust uitgebarsten',
 ' In plaats van de gron den

In [12]:
parser = etree.XMLParser(recover=True)

url= f"http://resolver.kb.nl/resolve?{file_name}"
response = requests.get(url)
if response.status_code != 200:
    print(f"URL: {url} is not valid.")

page_tree = response.content
page_tree = etree.fromstring(page_tree, parser=parser)

blocks = create_block_of_text(page_tree)

with open(f"{file_name}.txt", "w") as f:
    f.close()

# chunked_blocks = []
# for block in blocks:
#     if len(block["text"]) > 350:
#         chunks = split_long_text(block["text"])
#         for chunk in chunks:
#             chunked_blocks.append({"id": block["id"],
#                                 "wc_score": block["wc_score"],
#                                 "text": chunk})
            
# blocks = chunked_blocks if chunked_blocks else blocks

for block in blocks:
    print(f"ID: {block['id']}, Text: {block['text']}, WC Score: {round(block['wc_score'], 2)}")
    entities = model.predict_entities(block["text"], labels)

    # print(entities[0])
    for entity in entities:
        print(entity["text"], "=>", entity["label"])
    print("\n\n")

    tokens = block["text"].split(" ")
    token_offsets = [ (entity["start"], entity["end"]) for entity in entities]

    # print(f"Tokens: {tokens}")
    # print(f"Token count: {len(tokens)}")
    # print(f"Entities: {entities}")
    
    with open(f"{file_name}.txt", "a+") as f:
        token_labels = spans_to_bio(tokens, entities)
        
        for token_info in token_labels.values():
            f.write(f"{token_info['token']}\tPOS\t{token_info['label']}\n")

ID: P2_TB00001, Text: Koning zeer ongunstige tijdingen ontvangen ; zlj te vatten de mededeelinp', dat de levenskrachten sterk afnemen, dat er een algemeen lijdende toestand is ingetreden, terwijl zich eene groote verzwakking in de beenen heeft voorgedaan. Dientengevolge is voorloopig tegenbevel len aanzien der terugkomst van het kasteel der Ardennen naar hier gegeven " Eene korre«pondentie uit Rome deelt de instrukliën mede welke het revolutionnair zoogenaamd Romeinsch komité tot zijne aanhanaers geriït heeft Deze in strukliën moeten de oogen der ongeloovigsten doen opengaan, daar zij voor elkeen duidelijk doen uitko men dat de Italiaansche revolulionnairen voornemens zijn het Pauselijk gezag te vernietigen zoodra de Fransche troepen uit Rome zullen vertrokken zijn. Hoe meer de dag van de opening der zitting van de. Italiaansche Kamers nadert, hoe meer de ongele genheden aangro*ijen onder de voeten van het minis terie. He generaal La Marmora, minister-president, had, dm 12en, nog geene

/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 795 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Koning => person
Romeinsch komité => organization
Italiaansche revolulionnairen => organization
Pauselijk gezag => organization
generaal La Marmora => person
markies Gino Caypmi => person
graaf Casati => person
Kamer van Afgevaardigden => organization



ID: P2_TB00002, Text: j gaande nog niets beslist. Lord Elcho heeft de be noeming voorgesteld van eene kommissie, welke dit belangrijk vraagstuk zou bestudeeren. Buiten dit vraagstuk hield de openbare meening zich vooral bezig met de gebeurtenissen van Canada en Jamaica. De Mor ning Post hoopt dat Amerika He toebereidselen van eenen inval in Canada door de fenéans, zal verhinderen. Van een anderen kant verwachtte men met veel angst te Londen de aan komst van de mail, welke tijdingen uit Jamaica zal medebrengen. Met de in Southampton aangekomen Tasmanian zijn berigten uit Jamaica ontvangen, volgens welke de uitbarsting van den opstand voorbarig was geweest en eerst met Kersmis had moeten plaatshebben. De opstand werd met kracht onderdruk

/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 611 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Lord Elcho => person
Mor ning Post => organization
Tasmanian => person
westelijke kust => location (geographical)
Spaansche admiraal Pareja => person
Japansche regering => organization



ID: P2_TB00003, Text: AMERIKA, WC Score: 1.0
AMERIKA => location (geographical)



ID: P2_TB00004, Text: Men meldt uit New-York het overlijden van gene raal Alexander Schimmelpenninck van der Oije, een der bekwaamste generaals van het leger der Noorde lijken. Hij was in Litthauen geboren, had als officier in Pruissen en Baden gediend, was later in Ameri kaansche dienst overgegaan en daar tot den rang van generaal opgeklommen. In het voorjaar heeft hij Charleston ingenomen. Hij heeft vooral bij de be legering van die plaats ziju buitengewoon militair ta lent aan den dag gelegd. Men zegt dat hij zich van alle politieke inrigten vetwfjderd hield, wars was van zelfzucht, ten gevolge waarvan hij dan ook zijne vrouw en drie kinderen zonder vermogen nalaat. De weduwe kan hoogstens 30 dollars pensioen per maa

In [13]:
import pandas as pd
true_path = '../../../DATA/europeans-news/bio/'
pred_path = "."

true_df = pd.read_csv(os.path.join(true_path, f"{file_name}".replace(':', '_')+'.alto.xml.bio'), sep=' ', header=None, on_bad_lines='skip', engine='python')
pred_df = pd.read_csv(os.path.join(pred_path, f"{file_name}.txt"), sep='\t', header=None, on_bad_lines='skip', engine='python')

true_tags = [tag for tag in true_df.iloc[:, 2]]
pred_tags = [tag for tag in pred_df.iloc[:, 2]]

In [14]:
from seqeval.metrics import classification_report, f1_score


if len(true_tags) != len(pred_tags):
    print(f"Warning: true tags and predicted tags have different lengths ({len(true_tags)} vs {len(pred_tags)}). Truncating to the shorter length.")
    min_len = min(len(true_tags), len(pred_tags))
    true_tags = true_tags[:min_len]
    pred_tags = pred_tags[:min_len]

print(classification_report([true_tags], [pred_tags]))
print("\nF1 Score (micro avg):", f1_score([true_tags], [pred_tags]))

# Also print as dict for detailed metrics
report_dict = classification_report([true_tags], [pred_tags], output_dict=True)
print("\n" + "=" * 50)
print("DETAILED METRICS")
print("=" * 50)
for entity_type in sorted([k for k in report_dict.keys() if k not in ['micro avg', 'macro avg', 'weighted avg']]):
    metrics = report_dict[entity_type]
    print(f"{entity_type}:")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1-Score: {metrics['f1-score']:.4f}")

              precision    recall  f1-score   support

         LOC       0.90      0.45      0.60        20
         ORG       0.11      0.50      0.18         2
         PER       0.75      0.67      0.71         9

   micro avg       0.59      0.52      0.55        31
   macro avg       0.59      0.54      0.50        31
weighted avg       0.81      0.52      0.60        31


F1 Score (micro avg): 0.5517241379310345

DETAILED METRICS
LOC:
  Precision: 0.9000
  Recall: 0.4500
  F1-Score: 0.6000
ORG:
  Precision: 0.1111
  Recall: 0.5000
  F1-Score: 0.1818
PER:
  Precision: 0.7500
  Recall: 0.6667
  F1-Score: 0.7059


In [10]:
for t, p, true_token, pred_token in zip(true_tags, pred_tags, true_df.iloc[:, 0], pred_df.iloc[:, 0]):
    print(f"True: {t}, Pred: {p}, {true_token}, {pred_token}")

True: O, Pred: B-PER, Koning, Koning
True: O, Pred: O, zeer, zeer
True: O, Pred: O, ongunstige, ongunstige
True: O, Pred: O, tijdingen, tijdingen
True: O, Pred: O, ontvangen, ontvangen
True: O, Pred: O, ;, ;
True: O, Pred: O, zlj, zlj
True: O, Pred: O, te, te
True: O, Pred: O, vatten, vatten
True: O, Pred: O, de, de
True: O, Pred: O, mededeelinp',, mededeelinp',
True: O, Pred: O, dat, dat
True: O, Pred: O, de, de
True: O, Pred: O, levenskrachten, levenskrachten
True: O, Pred: O, sterk, sterk
True: O, Pred: O, afnemen,, afnemen,
True: O, Pred: O, dat, dat
True: O, Pred: O, er, er
True: O, Pred: O, een, een
True: O, Pred: O, algemeen, algemeen
True: O, Pred: O, lijdende, lijdende
True: O, Pred: O, toestand, toestand
True: O, Pred: O, is, is
True: O, Pred: O, ingetreden,, ingetreden,
True: O, Pred: O, terwijl, terwijl
True: O, Pred: O, zich, zich
True: O, Pred: O, eene, eene
True: O, Pred: O, groote, groote
True: O, Pred: O, verzwakking, verzwakking
True: O, Pred: O, in, in
True: O, Pred: